In [1]:
import csv

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

2026-03-05 00:38:36.011258: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-05 00:38:36.011483: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-05 00:38:36.043397: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-05 00:38:36.932536: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

# Specify each path

In [2]:
dataset = 'model/keypoint_classifier/keypoint.csv'
model_save_path = 'model/keypoint_classifier/keypoint_classifier.keras'

tflite_save_path = 'model/keypoint_classifier/keypoint_classifier.tflite'

# Set number of classes

In [3]:
NUM_CLASSES = 5

# Dataset reading
Ye wo data hai jo AI dekhega (Hath ke 21 points ke coordinates).

In [4]:
X_dataset = np.loadtxt(dataset, delimiter=',', dtype='float32', usecols=list(range(1, (21 * 2) + 1)))

# Dataset reading
$X$ = Sawal (Hath ke joints ki position/coordinates).
$y$ = Jawab (Us gesture ka naam ya ID).

In [5]:
y_dataset = np.loadtxt(dataset, delimiter=',', dtype='int32', usecols=(0))

# Spliting the DATA

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.75, random_state=RANDOM_SEED)

# Model building

In [7]:
model = tf.keras.models.Sequential([
    tf.keras.Input(shape=(21 * 2, )), #Mere model mein ek baar mein 42 numbers (21 points ke x,y) ek line mein lag kar andar aayenge.
    tf.keras.layers.Dropout(0.2), # 20 percent dropout
    tf.keras.layers.Dense(20, activation='relu'), #20 neurons in the first hidden layer with ReLU activation
    tf.keras.layers.Dropout(0.4), # 40 percent dropout
    tf.keras.layers.Dense(10, activation='relu'), #10 neurons in the second hidden layer with ReLU activation
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax') #Output layer with softmax activation
])

E0000 00:00:1772667518.797883   74312 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1772667518.805864   74312 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [8]:
model.summary()  # tf.keras.utils.plot_model(model, show_shapes=True)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dropout (Dropout)               │ (None, 42)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 20)             │           860 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           210 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │            55 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,125 (4.39 KB)

 Trainable params: 1,125 (4.39 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Model checkpoint callback
# To save weights after each epoch
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    model_save_path, verbose=1, save_weights_only=False)
# Callback for early stopping
# 20 round tak koi sudhaar nahi hua to training rok dega
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

In [10]:
# Model compilation
# Loss Function - Ki kitni badi glti hui hai (sparse_categorical_crossentropy) - kyunki humare labels integer form mein hain
#Optimizer - us galti ko theek kese karna hai (adam) - ek popular optimization algorithm
# Metrics - accuracy (hum model ki accuracy dekhna chahte hain)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Model training

In [11]:
model.fit(
    X_train,
    y_train,
    epochs=1000,
    batch_size=128,
    validation_data=(X_test, y_test),
    callbacks=[cp_callback, es_callback]
)

Epoch 1/1000
 1/31 ━━━━━━━━━━━━━━━━━━━━ 16s 549ms/step - accuracy: 0.2266 - loss: 1.6845
Epoch 1: saving model to model/keypoint_classifier/keypoint_classifier.keras
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.2711 - loss: 1.5673 - val_accuracy: 0.4822 - val_loss: 1.4294
Epoch 2/1000
 1/31 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3516 - loss: 1.5142
Epoch 2: saving model to model/keypoint_classifier/keypoint_classifier.keras
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3898 - loss: 1.4357 - val_accuracy: 0.5368 - val_loss: 1.3073
Epoch 3/1000
 1/31 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3828 - loss: 1.4228
Epoch 3: saving model to model/keypoint_classifier/keypoint_classifier.keras
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4242 - loss: 1.3492 - val_accuracy: 0.5915 - val_loss: 1.2008
Epoch 4/1000
 1/31 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3828 - loss: 1.3456
Epoch 4: saving model to model/keypoint_classifier/keypoint_classifier.ker

In [12]:
# Model evaluation
val_loss, val_acc = model.evaluate(X_test, y_test, batch_size=128)


11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9544 - loss: 0.2584 


In [13]:
# Loading the saved model
model = tf.keras.models.load_model(model_save_path)

In [14]:
# Inference test
predict_result = model.predict(np.array([X_test[0]]), verbose=0)
print(np.squeeze(predict_result))
print(np.argmax(np.squeeze(predict_result)))

[7.5075448e-02 3.7242122e-02 1.6954722e-02 1.5908675e-04 8.7056863e-01]
4


# Confusion matrix

In [15]:
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt
# from sklearn.metrics import confusion_matrix, classification_report

# def print_confusion_matrix(y_true, y_pred, report=True):
#     labels = sorted(list(set(y_true)))
#     cmx_data = confusion_matrix(y_true, y_pred, labels=labels)
    
#     df_cmx = pd.DataFrame(cmx_data, index=labels, columns=labels)
 
#     fig, ax = plt.subplots(figsize=(7, 6))
#     sns.heatmap(df_cmx, annot=True, fmt='g' ,square=False)
#     ax.set_ylim(len(set(y_true)), 0)
#     plt.show()
    
#     if report:
#         print('Classification Report')
#         print(classification_report(y_test, y_pred))

# Y_pred = model.predict(X_test)
# y_pred = np.argmax(Y_pred, axis=1)

# print_confusion_matrix(y_test, y_pred)

# Convert to model for Tensorflow-Lite

In [16]:
# Save as a model dedicated to inference
model.save(model_save_path, include_optimizer=False)

In [17]:
# Transform model (quantization)

#TFLite basically ek special format hai jo mobile processors ke liye optimize hota hai.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Quantization krta hai ye line
#Mtlb weights ko 8-bit integers mein convert krega from 32-bit float.
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# Ye line model ko quantize karke TFLite format mein convert kar degi.
tflite_quantized_model = converter.convert()

#Save the quantized model to a file
# open(tflite_save_path, 'wb').write(tflite_quantized_model)
with open(tflite_save_path, 'wb') as f:
    f.write(tflite_quantized_model)

INFO:tensorflow:Assets written to: /tmp/tmpl6lgkh8v/assets


INFO:tensorflow:Assets written to: /tmp/tmpl6lgkh8v/assets


Saved artifact at '/tmp/tmpl6lgkh8v'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 42), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  137207030047568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137205885444240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137205885450752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137205886281232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137205886272432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137205886279824: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1772667539.826056   74312 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1772667539.826073   74312 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-03-05 00:38:59.826313: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpl6lgkh8v
2026-03-05 00:38:59.826685: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-05 00:38:59.826690: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpl6lgkh8v
I0000 00:00:1772667539.829618   74312 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-03-05 00:38:59.830279: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-05 00:38:59.851143: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpl6lgkh8v
2026-03-05 00:38:59.856446: I tensorflow/cc/saved_model/loader.cc:471] SavedModel 

# Inference test

In [18]:
#TFLite model ko "chalaane" wala software engine.
interpreter = tf.lite.Interpreter(model_path=tflite_save_path)
#Memory allocate krta hai ye line, basically model ke liye jagah banata hai.
interpreter.allocate_tensors()

/home/dikki/Desktop/hand-gesture/hand_gesture_recon/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [19]:
# Get I / O tensor
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [20]:
interpreter.set_tensor(input_details[0]['index'], np.array([X_test[0]]))

In [21]:
%%time
# Inference implementation
interpreter.invoke() # Model ko chala raha hai
tflite_results = interpreter.get_tensor(output_details[0]['index'])

CPU times: user 166 μs, sys: 21 μs, total: 187 μs
Wall time: 134 μs


In [22]:
print(np.squeeze(tflite_results))
print(np.argmax(np.squeeze(tflite_results)))

[7.5075448e-02 3.7242122e-02 1.6954741e-02 1.5908689e-04 8.7056863e-01]
4
